- https://magazine.sebastianraschka.com/p/the-big-llm-architecture-comparison
    - qwen3 是一个全家桶，size * {dense, moe} * {vl, instruct}...
        - qwen3-32b, qwen3-30b-a3b 
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/11_qwen3/standalone-qwen3-moe.ipynb
    - https://mp.weixin.qq.com/s/G37CfOMRukGlAFhORptXzg

In [1]:
from IPython.display import Image

In [3]:
# https://mp.weixin.qq.com/s/G37CfOMRukGlAFhORptXzg
# https://huggingface.co/Qwen/Qwen3-30B-A3B/blob/main/config.json
# https://huggingface.co/Qwen/Qwen3-32B/blob/main/config.json
Image(url='./figs/qwen-moe.webp', width=500)

- layers (max_window_layers)
    - dense: num_hidden_layers=64
    - MoE: num_hidden_layers=48
- d_model 与注意力
    - dense: hidden_size=5120, num_attention_heads=64, head_dim=128
        - hidden_size ≠ num_attention_heads * head_dim
            - W_q: `[hidden_size, num_heads * head_dim]   = [5120, 8192 (64*128)]`
        - num_key_value_heads: GQA
            - 每 64 / 8 = 8 个 Query 头会共享同一组 Key 和 Value 头。
    - MoE: hidden_size=2048, num_attention_heads=32, head_dim=128
- FFN/MLP 中间维（dense 的 intermediate_size）
    - dense: intermediate_size=25600
        - 5\*5120(d\_model)
    - MoE: 
        - moe_intermediate_size = 768
            - 廋专家：“专家更瘦来控参控算”
        - 等效“dense 宽度”可用公式：$d_{ff,equiv}=k\times m=8\times 768$（指聚合后的等效宽度，而非单个 expert 的宽度）
            - intermediate_size=6144=3*2048(d\_model)，每个 expert 的中间维度 d\_ff\_expert)
            - intermediate_size = 6144：这是传统 dense MLP（非 MoE）的中间宽度，属于“兼容/占位”配置；当某层是普通 MLP 时才会用到。在这个 A3B 检查点里，`decoder_sparse_step=1` 且 `mlp_only_layers=[]`，意思是每一层都是 MoE 层，因此 6144 基本不参与前向（除非你改成部分层 dense）。
                - decoder_sparse_step 决定解码器里 MoE（稀疏专家）层的“步长/频率”。当且仅当 (layer_idx % decoder_sparse_step) == 0 的那些层会用 MoE-MLP，否则就是普通 dense-MLP。

> 关于等效宽度（top-8 的 experts 不是概率加权求和吗？不是 concat 啊）
- flops 计算
    - dense：up，gate，down，flops = 3 * d_model * d_ff
    - moe: 每个被选中的专家各做一次 up、gate、down，合计 k 次，每次宽度 m
        - flops = 3 * d_model * k * m
    - 两者拉齐，就是 d_ff = k * m，这是等效宽度的来源
    - 单个的mlp（一个expert）的表达能力是被降低了，用多experts来承载多样性。
- 矩阵 rank 的角度：“加权求和”可以等价改写成“concat 后乘一个按块稀疏的矩阵
    - $v_i\in\mathbb R^d,\alpha_i\in\mathbb R$
    - weighted sum：$\sum_{i=1}^k\alpha_iv_i$
    - 稀疏矩阵的写法
        - $A(\alpha)=\begin{bmatrix}\alpha_1I_d & \alpha_2I_d & \cdots & \alpha_kI_d\end{bmatrix}\in \mathbb R^{d\times kd}$
        - $A(\alpha)\begin{bmatrix} v_1\\ v_2\\ \vdots\\ v_k \end{bmatrix}
= \sum_{i=1}^k \alpha_i I_d\, v_i
= \sum_{i=1}^k \alpha_i v_i .$ (对 $v_i$ 纵向 concat，维度为 $kd$)

In [10]:
import numpy as np

v1 = np.array([1., 2.])
v2 = np.array([3., 4.])
v3 = np.array([-1., 5.])
alpha = np.array([0.2, 0.3, 0.5])  # weights

V_concat = np.concatenate([v1, v2, v3])  # shape (k*d,)
d = 2
I = np.eye(d)
A = np.kron(alpha.reshape(1, -1), I)  # shape (d, k*d)

y_matrix = A @ V_concat
y_sum = alpha[0]*v1 + alpha[1]*v2 + alpha[2]*v3

y_matrix, y_sum

(array([0.6, 4.1]), array([0.6, 4.1]))

- $k=3,d=2$
- $v_1=\begin{bmatrix}1\\ 2\end{bmatrix},v_2=\begin{bmatrix}3\\ 4\end{bmatrix},v_3=\begin{bmatrix}-1\\ 5\end{bmatrix}$
- $\alpha=(0.2, 0.3, 0.5)$
- $A(\alpha)=\begin{bmatrix}0.2 & 0 & 0.3 & 0 & 0.5 & 0\\ 0 & 0.2 & 0 & 0.3 & 0 & 0.5\end{bmatrix}$
- $\begin{bmatrix}v_1\\v_2\\v_3\end{bmatrix}=\begin{bmatrix}1\\2\\3\\4\\-1\\5\end{bmatrix}$
- $A(\alpha)\begin{bmatrix}v_1\\v_2\\v_3\end{bmatrix}=\sum_i\alpha_iv_i$

| 参数 (Parameter)                 | Qwen3-30B-A3B (MoE) | Qwen3-32B (Dense) |
|----------------------------------|---------------------|-------------------|
| architectures                    | Qwen3MoeForCausalLM | Qwen3ForCausalLM  |
| model_type                       | qwen3_moe           | qwen3             |
| hidden_size                      | 2048                | 5120              |
| head_dim                         | 128                 | 128               |
| num_hidden_layers                | 48                  | 64                |
| num_attention_heads              | 32                  | 64                |
| num_key_value_heads              | 4                   | 8                 |
| intermediate_size (dense MLP)    | 6144                | 25600             |
| vocab_size                       | 151936              | 151936            |
| max_position_embeddings          | 40960               | 40960             |
| max_window_layers                | 48                  | 64                |
| rope_theta                       | 1,000,000           | 1,000,000         |
| rope_scaling                     | null                | null              |
| attention_bias                   | false               | false             |
| attention_dropout                | 0.0                 | 0.0               |
| rms_norm_eps                     | 1e-6                | 1e-6              |
| tie_word_embeddings              | false               | false             |
| torch_dtype                      | bfloat16            | bfloat16          |
| use_cache                        | true                | true              |
| sliding_window / use_sliding_window | null / false     | null / false      |
| **MoE: num_experts**             | 128                 | —                 |
| **MoE: num_experts_per_tok**     | 8                   | —                 |
| **MoE: moe_intermediate_size**   | 768                 | —                 |
| **MoE: decoder_sparse_step**     | 1                   | —                 |
| **MoE: norm_topk_prob**          | true                | —                 |
| **MoE: router_aux_loss_coef**    | 0.001               | —                 |
| **MoE: output_router_logits**    | false               | —                 |


In [5]:
Image(url='./figs/representation.png', width=200)

- d_model 的一致性，方便残差实现
- moe/ffn: token level
    - https://www.bilibili.com/opus/942536178060492803?spm_id_from=333.1387.0.0
    - attention: token interaction
    - ffn(moe): token projection

### 参数量计算

- embedding: 151_936 × 2048 * 2
    - tie_word_embeddings: false

$$
\text{Attn}_h = \text{softmax}\left(\frac{Q_h K_{g(h)}^\top}{\sqrt{128}}\right) V_{g(h)}
$$

- gqa: `head_dim = 128, num_attention_heads = 32, num_key_value_heads = 4`
    - 32 / 4 = 8: 每 8 个 Q 头共享同一组 K/V 头 (一共4组, `n_kv_groups = 4`)
        - `[0..7], [8..15], [16..23], [24..31]`
    - W_q: 2048\*(32\*128)=2048×4096,
    - W_k: 2048\*(4\*128), W_v: 2048\*(4\*128),
    - W_o: 4096\*2048

$$
H = \text{SiLU}(XW_{\text{up}}) \odot (XW_{\text{gate}}), \quad Y = HW_{\text{down}}
$$

- moe: `moe_intermediate_size = 768`
    - 路由：2048×128 = 262,144
        - top 8 scores
        - top 8 scores => softmax, gating coef 
    - 每位专家 (ffn)：2048×768（up） + 2048×768（gate） + 768×2048（down） = 3×(2048×768) = 4,718,592
        - top8 experts 基于 gating coef 进行加权求和
    - `norm_topk_prob=true`
        - 先计算 128 scores
        - top 8
        - softmax

In [9]:
# 30b-a3b
# https://huggingface.co/Qwen/Qwen3-30B-A3B
151_936 * 2048 * 2 + 48 * ((2048*4096 + 2048*512 + 2048 * 512 + 4096 * 2048) + 2048 * 128 + 128 * (2048*768*3))

30531911680

In [8]:
# gqa, ffn(moe)
48 * ((2048*4096 + 2048*512 + 2048 * 512 + 4096 * 2048)) / 30531911680, 48 * (2048 * 128 + 128 * (2048*768*3)) / 30531911680

(0.02967287713574311, 0.9499441916373315)

In [6]:
# activation
151_936 * 2048 * 2 + 48 * ((2048*4096 + 2048*512 + 2048 * 512 + 4096 * 2048) + 2048 * 128 + 8 * (2048*768*3))

3352821760

In [9]:
# activation, gqa, ffn(moe)
48 * ((2048*4096 + 2048*512 + 2048 * 512 + 4096 * 2048)) / 3352821760, 48 * (2048 * 128 + 8 * (2048*768*3)) / 3352821760

(0.27021110242376856, 0.544175136825645)

In [11]:
# dense 32b
151936 * 5120 * 2 + 64 * (5120 * (64 * 128) + 5120 * (8 * 128) * 2  + (64 * 128) * 5120 + 5120 * 25600 * 3)

32761446400

In [12]:
# dense 32b
64 * (5120 * (64 * 128) + 5120 * (8 * 128) * 2 + (64 * 128) * 5120) /  32761446400, 64 * (5120 * 25600 * 3) / 32761446400

(0.18435687137427487, 0.7681536307261452)

https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/11_qwen3/standalone-qwen3.ipynb
$$
\text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}, \qquad \mathbf{y} = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x})} \odot \gamma
$$
- 忽略了rmsnorm 的参数 (不减均值（与 LayerNorm $\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}\gamma+\beta$ 不同），只按幅度（RMS）归一，再做逐维缩放。)
    - 层内，2个 rmsnorm ($\gamma$)，2 * d_model,
    - 以及末尾的 d_model: 1 * d_model

In [14]:
32761446400 + 64 * 2 * 5120 + 5120

32762106880

### transformers 源码

- https://github.com/huggingface/transformers/tree/main/src/transformers/models
    - qwen3, qwen3-moe, qwen3-vl, qwen3-vl-moe
- https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_moe/modeling_qwen3_moe.py#L226-L264
    - 按 expert 分桶 + index_add_，只算命中的 token → 省显存/省 FLOPs。
    - https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_vl_moe/modeling_qwen3_vl_moe.py#L97-L145
        - eval mode ??

### coding

- 先取路由的 Top-K 专家索引 $I_t=\{e_{t,1},\cdots, e_{t,K}\}$（$t$ 内专家一定不重复） 和对应的 softmax 概率 $P_t=\{p_{t,1},\cdots,p_{t,K}\}$
    - 这是只在 Top-K 内归一的 softmax，$\sum_{j=1}^Kp_{e,j}=1$
- 每个被选中的专家 $e_{t,j}$ 给出的该 token 的前馈输出 $F_{e_{t,j}}(x_t) := \text{fc3}_e(\text{silu}(\text{fc1}_e(x_t)) \odot \text{fc2}_e(x_t)).$
- 最终输出就是按概率的线性加权和：
    - $y_t=\sum_{j=1}^Kp_{t,j}F_{e,j}(x_t)$

In [10]:
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

huggingface_hub version: 0.33.4
tokenizers version: 0.21.2
torch version: 2.6.0


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [33]:
class MoEFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_experts_per_tok = cfg["num_experts_per_tok"]
        self.num_experts = cfg["num_experts"]
        self.emb_dim = cfg["emb_dim"]
        self.gate = nn.Linear(cfg["emb_dim"], cfg["num_experts"], bias=False, dtype=cfg["dtype"])

        # up
        self.fc1 = nn.ModuleList([nn.Linear(cfg["emb_dim"], cfg["moe_intermediate_size"], bias=False, dtype=cfg["dtype"])
                                  for _ in range(cfg["num_experts"])])
        # gate 
        self.fc2 = nn.ModuleList([nn.Linear(cfg["emb_dim"], cfg["moe_intermediate_size"], bias=False, dtype=cfg["dtype"])
                                  for _ in range(cfg["num_experts"])])
        # down
        self.fc3 = nn.ModuleList([nn.Linear(cfg["moe_intermediate_size"], cfg["emb_dim"], bias=False, dtype=cfg["dtype"])
                                  for _ in range(cfg["num_experts"])])

    def forward(self, x):
        scores = self.gate(x)  # (b, seq_len, num_experts)
        topk_scores, topk_indices = torch.topk(scores, self.num_experts_per_tok, dim=-1)
        topk_probs = torch.softmax(topk_scores, dim=-1)

        batch, seq_len, _ = x.shape
        x_flat = x.reshape(batch * seq_len, -1)
        out_flat = torch.zeros(batch * seq_len, self.emb_dim, device=x.device, dtype=x.dtype)

        topk_indices_flat = topk_indices.reshape(-1, self.num_experts_per_tok)
        topk_probs_flat = topk_probs.reshape(-1, self.num_experts_per_tok)

        unique_experts = torch.unique(topk_indices_flat)

        for expert_id_tensor in unique_experts:
            expert_id = int(expert_id_tensor.item())
            mask = topk_indices_flat == expert_id
            if not mask.any():
                continue

            token_mask = mask.any(dim=-1)
            selected_idx = token_mask.nonzero(as_tuple=False).squeeze(-1)
            if selected_idx.numel() == 0:
                continue

            expert_input = x_flat.index_select(0, selected_idx)
            hidden = torch.nn.functional.silu(self.fc1[expert_id](expert_input)) * self.fc2[expert_id](expert_input)
            expert_out = self.fc3[expert_id](hidden)

            mask_selected = mask[selected_idx]
            slot_indices = mask_selected.int().argmax(dim=-1, keepdim=True)
            selected_probs = torch.gather(topk_probs_flat.index_select(0, selected_idx), dim=-1, index=slot_indices).squeeze(-1)

            out_flat.index_add_(0, selected_idx, expert_out * selected_probs.unsqueeze(-1))

        return out_flat.reshape(batch, seq_len, self.emb_dim)

In [13]:
cfg = {
    "num_experts_per_tok": 2,   # top-2 routing (per token)
    "num_experts": 3,           # 3 experts: E0, E1, E2
    "emb_dim": 2,               # token embedding dim
    "moe_intermediate_size": 3, # FFN hidden per expert
    "dtype": torch.float64,
}
moe = MoEFeedForward(cfg)

In [24]:
# Tiny input: batch=1, seq=3 tokens
# Intuition:
#  - t0 = [2,0]  => gate scores [2,0,2] => top2 = {E0, E2}, equal => probs ~ [0.5, 0.5]
#  - t1 = [0,3]  => gate scores [0,3,3] => top2 = {E1, E2}, equal => probs ~ [0.5, 0.5]
#  - t2 = [1,1]  => gate scores [1,1,2] => top2 = {E2, E0 or E1}; by value it's {E2, E0} (tie between E0/E1, topk keeps lowest index)
x = torch.tensor([[[2.0, 0.0],
                   [0.0, 3.0],
                   [1.0, 1.0]]], dtype=torch.float64)  # shape (1,3,2)

print("=== Input x (batch=1, seq=3, emb_dim=2) ===")
print(x, "\n")

=== Input x (batch=1, seq=3, emb_dim=2) ===
tensor([[[2., 0.],
         [0., 3.],
         [1., 1.]]], dtype=torch.float64) 



- $x$: (1, 3, 2)
- $w_{g}$: (3, 2), 3 experts
    - y = x W^T = (1, 3, 3)
- top2 & softmax
    - (1, 3, 2): row sum = 1

In [14]:
# Gate: scores = x @ W_gate^T  (since nn.Linear: y = x W^T)
W_gate = torch.tensor([
    [1.0, 0.0],  # E0 score = x0*1 + x1*0
    [0.0, 1.0],  # E1 score = x0*0 + x1*1
    [1.0, 1.0],  # E2 score = x0*1 + x1*1
], dtype=torch.float64)  # shape (num_experts, emb_dim)
with torch.no_grad():
    moe.gate.weight.copy_(W_gate)

# Expert weights
# For clarity, fc3 for all experts just "picks" the first two hidden dims (ignore the 3rd dim).
W_fc3 = torch.tensor([
    [1.0, 0.0, 0.0],  # output dim 0
    [0.0, 1.0, 0.0],  # output dim 1
], dtype=torch.float64)  # shape (emb_dim, moe_intermediate_size)

# Expert 0, e0
W_fc1_e0 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
], dtype=torch.float64)  # (3,2)
W_fc2_e0 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0,-1.0],
], dtype=torch.float64)  # (3,2)

# Expert 1, e1
W_fc1_e1 = torch.tensor([
    [2.0, 0.0],
    [0.0, 0.5],
    [1.0, 1.0],
], dtype=torch.float64)
W_fc2_e1 = torch.tensor([
    [0.5, 0.0],
    [0.0, 1.5],
    [1.0, 0.0],
], dtype=torch.float64)

# Expert 2, e2
W_fc1_e2 = torch.tensor([
    [0.5, 0.5],
    [1.0, 0.0],
    [0.0, 1.0],
], dtype=torch.float64)
W_fc2_e2 = torch.tensor([
    [1.0, 1.0],
    [0.0, 2.0],
    [2.0, 0.0],
], dtype=torch.float64)

with torch.no_grad():
    for e, (W1, W2) in enumerate([(W_fc1_e0, W_fc2_e0),
                                  (W_fc1_e1, W_fc2_e1),
                                  (W_fc1_e2, W_fc2_e2)]):
        moe.fc1[e].weight.copy_(W1)
        moe.fc2[e].weight.copy_(W2)
        moe.fc3[e].weight.copy_(W_fc3)

In [25]:
# ---------- Forward pass (module) ----------
y = moe(x)
print("=== Module forward() output (y) ===")
print(y, "\n")

=== Module forward() output (y) ===
tensor([[[2.4927, 0.0000],
         [1.8395, 2.7593],
         [1.2655, 1.2655]]], dtype=torch.float64, grad_fn=<ViewBackward0>) 



In [31]:
with torch.no_grad():
    scores = moe.gate(x)
    topk_scores, topk_indices = torch.topk(scores, moe.num_experts_per_tok, dim=-1)
    topk_probs = torch.softmax(topk_scores, dim=-1)             # (B,L,K)
    B, L, D = x.shape
    K = moe.num_experts_per_tok

    # 手工重构输出
    recon = torch.zeros_like(x)
    for b in range(B):
        for t in range(L):
            for j in range(K):
                e = int(topk_indices[b,t,j])
                p = topk_probs[b,t,j]                            # 该 expert 的权重
                h = F.silu(moe.fc1[e](x[b,t])) * moe.fc2[e](x[b,t])
                f = moe.fc3[e](h)                                # 该 expert 对该 token 的输出
                recon[b,t] += p * f

In [32]:
y, recon

(tensor([[[2.4927, 0.0000],
          [1.8395, 2.7593],
          [1.2655, 1.2655]]], dtype=torch.float64, grad_fn=<ViewBackward0>),
 tensor([[[2.4927, 0.0000],
          [1.8395, 2.7593],
          [1.2655, 1.2655]]], dtype=torch.float64))

In [30]:
# ---------- Manual, step-by-step reproduction ----------
scores = moe.gate(x)  # (1,3,3)
print("Gate scores (before top-k):")
print(scores, "\n")

Gate scores (before top-k):
tensor([[[2., 0., 2.],
         [0., 3., 3.],
         [1., 1., 2.]]], dtype=torch.float64, grad_fn=<UnsafeViewBackward0>) 



In [17]:
topk_scores, topk_indices = torch.topk(scores, moe.num_experts_per_tok, dim=-1)
topk_probs = torch.softmax(topk_scores, dim=-1)

print("Top-k indices per token:")
print(topk_indices)
print("Top-k scores per token:")
print(topk_scores)
print("Top-k softmax probs per token (normalize within top-k):")
print(topk_probs, "\n")

Top-k indices per token:
tensor([[[0, 2],
         [1, 2],
         [2, 0]]])
Top-k scores per token:
tensor([[[2., 2.],
         [3., 3.],
         [2., 1.]]], dtype=torch.float64, grad_fn=<TopkBackward0>)
Top-k softmax probs per token (normalize within top-k):
tensor([[[0.5000, 0.5000],
         [0.5000, 0.5000],
         [0.7311, 0.2689]]], dtype=torch.float64, grad_fn=<SoftmaxBackward0>) 



In [18]:
batch, seq_len, _ = x.shape
x_flat = x.reshape(batch * seq_len, -1)
out_flat = torch.zeros(batch * seq_len, moe.emb_dim, dtype=x.dtype)

topk_indices_flat = topk_indices.reshape(-1, moe.num_experts_per_tok)
topk_probs_flat = topk_probs.reshape(-1, moe.num_experts_per_tok)

print("Flattened tokens (B*L, D):")
print(x_flat, "\n")
print("Flattened top-k indices (per token):")
print(topk_indices_flat)
print("Flattened top-k probs (per token):")
print(topk_probs_flat, "\n")

Flattened tokens (B*L, D):
tensor([[2., 0.],
        [0., 3.],
        [1., 1.]], dtype=torch.float64) 

Flattened top-k indices (per token):
tensor([[0, 2],
        [1, 2],
        [2, 0]])
Flattened top-k probs (per token):
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000],
        [0.7311, 0.2689]], dtype=torch.float64, grad_fn=<ViewBackward0>) 



In [19]:
unique_experts = torch.unique(topk_indices_flat)
print("Unique experts selected across the batch:", unique_experts.tolist(), "\n")

Unique experts selected across the batch: [0, 1, 2] 



In [23]:
# We'll also collect per-token contributions for clarity
per_token_contribs = {i: [] for i in range(batch*seq_len)}

for expert_id_tensor in unique_experts:
    expert_id = int(expert_id_tensor.item())
    mask = topk_indices_flat == expert_id            # (B*L, K) True where this expert is selected
    token_mask = mask.any(dim=-1)                    # (B*L,) which tokens select this expert
    selected_idx = token_mask.nonzero(as_tuple=False).squeeze(-1)
    if selected_idx.numel() == 0:
        continue

    expert_input = x_flat.index_select(0, selected_idx)
    h1 = moe.fc1[expert_id](expert_input)
    h2 = moe.fc2[expert_id](expert_input)
    hidden = F.silu(h1) * h2                         # GLU-style gating
    expert_out = moe.fc3[expert_id](hidden)          # (nsel, emb_dim)

    # for each selected token, pick its probability slot for this expert
    mask_selected = mask[selected_idx]               # (nsel, K)
    slot_indices = mask_selected.int().argmax(dim=-1, keepdim=True)  # position 0..K-1
    selected_probs = torch.gather(topk_probs_flat.index_select(0, selected_idx), dim=-1, index=slot_indices).squeeze(-1)

    contrib = expert_out * selected_probs.unsqueeze(-1)  # weighted contribution
    out_flat.index_add_(0, selected_idx, contrib)

    print(f"[Expert {expert_id}] selected token indices:", selected_idx.tolist())
    print(f"  expert_input:\n{expert_input}")
    print(f"  fc1(x) (pre-silu):\n{h1}")
    print(f"  fc2(x):\n{h2}")
    print(f"  hidden = silu(fc1(x)) * fc2(x):\n{hidden}")
    print(f"  expert_out = fc3(hidden):\n{expert_out}")
    print(f"  selected_probs ({moe.num_experts_per_tok}-way softmax within token):\n{selected_probs}")
    print(f"  contribution = expert_out * prob:\n{contrib}\n")

    for idx, c in zip(selected_idx.tolist(), contrib):
        per_token_contribs[idx].append((expert_id, c.detach().clone()))

print("Accumulated out_flat (B*L, D):")
print(out_flat, "\n")

y_manual = out_flat.reshape(batch, seq_len, moe.emb_dim)
print("=== Manual reconstruction y_manual ===")
print(y_manual, "\n")

print("Allclose(module_output, manual) ->", torch.allclose(y, y_manual))

# Pretty-print per-token decomposition
for tok in range(batch*seq_len):
    print(f"\n--- Token {tok} contributions (sum -> output token {tok}) ---")
    for (eid, c) in per_token_contribs[tok]:
        print(f"  Expert {eid} contribution: {c}")
    print(f"  Sum: {sum(c for _, c in per_token_contribs[tok])}")

[Expert 0] selected token indices: [0, 2]
  expert_input:
tensor([[2., 0.],
        [1., 1.]], dtype=torch.float64)
  fc1(x) (pre-silu):
tensor([[2., 0., 2.],
        [1., 1., 2.]], dtype=torch.float64, grad_fn=<MmBackward0>)
  fc2(x):
tensor([[2., 0., 2.],
        [1., 1., 0.]], dtype=torch.float64, grad_fn=<MmBackward0>)
  hidden = silu(fc1(x)) * fc2(x):
tensor([[3.5232, 0.0000, 3.5232],
        [0.7311, 0.7311, 0.0000]], dtype=torch.float64, grad_fn=<MulBackward0>)
  expert_out = fc3(hidden):
tensor([[3.5232, 0.0000],
        [0.7311, 0.7311]], dtype=torch.float64, grad_fn=<MmBackward0>)
  selected_probs (2-way softmax within token):
tensor([0.5000, 0.2689], dtype=torch.float64, grad_fn=<SqueezeBackward1>)
  contribution = expert_out * prob:
tensor([[1.7616, 0.0000],
        [0.1966, 0.1966]], dtype=torch.float64, grad_fn=<MulBackward0>)

[Expert 1] selected token indices: [1]
  expert_input:
tensor([[0., 3.]], dtype=torch.float64)
  fc1(x) (pre-silu):
tensor([[0.0000, 1.5000, 3.000